# FphIShE MVP on Colab

**Open in Colab:** https://colab.research.google.com/github/Eitzl/FphIShE/blob/feature/work-space/notebooks/run_mvp.ipynb

You can also use File → Open notebook → GitHub and pick `Eitzl/FphIShE`, branch
`feature/work-space` (tick "Include private repos" if the repo is private).

The full explanations are in `docs/colab_guide.md`. The section numbers match. Pipeline:
data → ProtT5 embeddings → queries → searches (fphishe, CPU SW, BLAST) → GPU check and GPU SW → report.

## 0. Colab setup

**0.1** Runtime → Change runtime type → any GPU. The free one is a T4; an A100 or L4 is faster.
Timings depend on the hardware, so `report.py` prints the GPU name and CPU count next to them.

In [ ]:
!nvidia-smi

**0.2** Clone. The plain clone works if the repo is public. If it fails with an authentication
error, use the token cell below instead.

In [ ]:
!git clone -b feature/work-space https://github.com/Eitzl/FphIShE.git /content/FphIShE
%cd /content/FphIShE
!git log --oneline -3

Private repo only: add a fine-grained GitHub token (Contents: Read-only on `Eitzl/FphIShE`) as
Colab secret `GH_TOKEN` (key icon, with Notebook access on). **Never paste the token into a cell or
commit it.** The second command removes the token from `.git/config`.

In [ ]:
from google.colab import userdata

TOKEN = userdata.get("GH_TOKEN")
!git clone -b feature/work-space https://$TOKEN@github.com/Eitzl/FphIShE.git /content/FphIShE
!git -C /content/FphIShE remote set-url origin https://github.com/Eitzl/FphIShE.git
%cd /content/FphIShE
!git log --oneline -3

Pull fixes later. Revert local edits first, for example `!git checkout -- scripts/embed.py`.
Use the token version of the pull for a private repo.

In [ ]:
!git -C /content/FphIShE pull origin feature/work-space

In [ ]:
TOKEN = userdata.get("GH_TOKEN")
!git -C /content/FphIShE pull https://$TOKEN@github.com/Eitzl/FphIShE.git feature/work-space

**0.4** Install. Run scripts as `!python scripts/...` from the repo root.

In [ ]:
%cd /content/FphIShE
!pip install -q -e ".[align,gpu,embed,dev]"
!apt-get -qq install -y ncbi-blast+

In [ ]:
!python -c "import fphishe, faiss, parasail, numba, torch, transformers, h5py; print('numba', numba.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!blastp -version
!pytest -q

**0.5** Mount Drive to keep `embeddings.h5` across sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/fphishe

## 1. Data

This should print `15177 sequences`.

In [ ]:
!python scripts/download_data.py

## 2. Embeddings

**2.1** New session only: restore embeddings saved by an earlier session.

In [ ]:
!mkdir -p data/processed
!cp -v /content/drive/MyDrive/fphishe/embeddings.h5 data/processed/ 2>/dev/null || echo "no saved embeddings"

**2.2** Embed. This is the slow step: measured 1.6 min on an A100, an estimated 10–30 min on a T4, plus a ~2.4 GB model download on
first run. It's resumable. Rerun after a crash. It must say `on cuda`. If it runs out of memory, run
`!sed -i 's/^MAX_RESIDUES = 4000/MAX_RESIDUES = 2000/' scripts/embed.py` and rerun.
Record the wall time.

In [ ]:
%%time
!python scripts/embed.py

**2.3** Check the count (expect 15177) and back up to Drive. Write locally first, then copy.

In [ ]:
!ls -lh data/processed/embeddings.h5
!python -c "import h5py; print(len(h5py.File('data/processed/embeddings.h5','r')), 'embedded (expect 15177)')"
!cp -v data/processed/embeddings.h5 /content/drive/MyDrive/fphishe/

## 3. Queries

Picking new queries makes all existing hits stale, so rerun every search afterwards.
Deleting `timings.tsv` gives a clean run.

In [ ]:
!python scripts/pick_queries.py
!rm -f results/timings.tsv

**3.1** Time what embedding a new query costs. The queries are already in `embeddings.h5`, so
fphishe's search time leaves this out. This embeds them again into a separate file and logs the
time as `embed`; the report adds it to fphishe's time as "end to end".

In [ ]:
!rm -f data/processed/query_embeddings.h5
!python scripts/embed.py --ids data/processed/queries.txt --out data/processed/query_embeddings.h5

## 4. Searches (CPU)

**4.1** FphIShE: PCA + IVF prefilter, then parasail rerank.

In [ ]:
!python scripts/search_fphishe.py

**4.2** Full Smith-Waterman on CPU (multiprocessing, so run it as a script). Measured: 23 s on the
A100 runtime; an estimated 2 min on a 2-vCPU T4 runtime.

In [ ]:
%%time
!python scripts/search_sw.py

**4.3** BLAST. The middle column of the hits file must be a bare domain id like `d1dlwa_`.

In [ ]:
!python scripts/search_blast.py
!head -3 results/hits_blast.tsv

## 5. GPU check, then GPU Smith-Waterman

Stop at the first failure and send Claude: the full output or traceback, the `cuda.detect()`
output, the numba and numba-cuda versions, and `nvidia-smi`. Don't set `NUMBA_ENABLE_CUDASIM` in
this notebook.

**5.1** Does numba see the GPU?

In [ ]:
!python -c "from numba import cuda; cuda.detect()"

If that failed at driver or NVVM init, install numba-cuda and rerun 5.1.

In [ ]:
!pip install -q "numba-cuda[cu12]"

**5.2** Kernel logic in the CUDA simulator (CPU).

In [ ]:
!pytest -q tests/test_gpu.py

**5.3** Real hardware. This must print `0 mismatches`. Record the JIT time, GCUPS, and speedup.

In [ ]:
!python scripts/check_gpu.py --queries 5 --check 300

**5.4** Full GPU search, only after 0 mismatches. The `sw_gpu` result should match `sw`.

In [ ]:
%%time
!python scripts/search_sw.py --gpu

**5.5** Versions to paste into a bug report.

In [ ]:
!python -c "import numba; print(numba.__version__)"
!pip list 2>/dev/null | grep -i -E "numba|cuda"
!nvidia-smi | head -4

## 6. Report

`--k` must match the fphishe `--k`. In the output, `prefilter` recall@k is the ceiling on fphishe.
`family` and `remote` split sensitivity into close homologs (same family) and remote ones (same
superfamily, different family). The `*_build` ms/item numbers aren't per-query costs. See the
guide for how to read the table.

In [ ]:
!python scripts/report.py

In [ ]:
from IPython.display import Image

Image("results/sensitivity.png")

Save the results to Drive.

In [ ]:
!cp -rv results /content/drive/MyDrive/fphishe/results_$(date +%Y%m%d_%H%M)

## 7. Experiments

Each one is a single flag: `--k`, `--dim`, `--nprobe`, `--metric cos` on `search_fphishe.py`, and
`--n` on `pick_queries.py` (then rerun all searches). `--repeats 3` on any search script reports
the median time. Example k sweep, then a plot of it:

In [ ]:
!for k in 25 50 100 200 400; do python scripts/search_fphishe.py --k $k && python scripts/report.py --k $k && cp results/summary.tsv results/summary_k$k.tsv; done

In [ ]:
import glob
import re

import matplotlib.pyplot as plt
import pandas as pd

ks, sens, recall = [], [], []
for path in sorted(glob.glob("results/summary_k*.tsv"), key=lambda p: int(re.findall(r"\d+", p)[-1])):
    s = pd.read_csv(path, sep="\t").set_index("method")
    ks.append(int(re.findall(r"\d+", path)[-1]))
    sens.append(s.loc["fphishe", "sens_mean"])
    recall.append(s.loc["prefilter"].filter(like="recall@").iloc[0])
plt.plot(ks, sens, "o-", label="fphishe sensitivity")
plt.plot(ks, recall, "s-", label="prefilter recall@k")
plt.xscale("log"); plt.xlabel("k"); plt.legend(); plt.show()

In [ ]:
# Other single-flag runs (uncomment one at a time, then rerun the report)
# !python scripts/search_fphishe.py --dim 256
# !python scripts/search_fphishe.py --nprobe 100   # exact search with nlist=100
# !python scripts/report.py

## 8. Troubleshooting

See the table in `docs/colab_guide.md` section 8.

## 9. Known rough edges and optimization candidates

See `docs/colab_guide.md` section 9.

## Questions to note as you go

| Step | Wall time | Problems / confusion |
|---|---|---|
| 0 Setup | | |
| 1 Data | | |
| 2 Embeddings (h5 size, MAX_RESIDUES) | | |
| 3 Queries | | |
| 4.1 search_fphishe (build / search) | | |
| 4.2 search_sw CPU | | |
| 4.3 search_blast | | |
| 5.3 check_gpu (mismatches / JIT / GCUPS / speedup) | | |
| 5.4 search_sw --gpu | | |
| 6 report | | |

Where did you have to guess? Which errors misled you? Is prefilter recall@k high enough? Did `sw_gpu` match `sw`?